# 00 — Environment and data setup

Mount Drive, clone the repository, install dependency groups, verify GPU/CUDA, create the standardized directory tree, and convert VisDrone without redistributing it. The VMamba cell uses its official repository and detection tree.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/visdrone_architecture_benchmark"
GITHUB_USERNAME = "Harryphan72007"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/aerial-object-detection-benchmark.git"
REPO_DIR = "/content/aerial-object-detection-benchmark"
!test -d {REPO_DIR}/.git || git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt
!pip install -q -e .

In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

## Install MMDetection dependencies

MMCV wheels must match the active PyTorch/CUDA image. `mim` chooses a compatible published wheel when available. Restart the runtime after binary-package changes.

In [ ]:
!pip install -q openmim
!mim install -q "mmcv>=2.0.0,<2.2.0"
!pip install -q "mmdet==3.3.0"

## Clone dependency repositories

The commit hashes are printed and later copied into run metadata. Do not edit the upstream source in-place without recording a patch.

In [ ]:
MMDET_ROOT = "/content/mmdetection"
VMAMBA_ROOT = "/content/VMamba"
!test -d $MMDET_ROOT/.git || git clone --depth 1 https://github.com/open-mmlab/mmdetection.git $MMDET_ROOT
!test -d $VMAMBA_ROOT/.git || git clone --depth 1 https://github.com/MzeroMiko/VMamba.git $VMAMBA_ROOT
%env MMDET_ROOT=$MMDET_ROOT
%env VMAMBA_ROOT=$VMAMBA_ROOT
!git -C $MMDET_ROOT rev-parse HEAD
!git -C $VMAMBA_ROOT rev-parse HEAD

## Convert and validate data

Place the official downloads under `DRIVE_ROOT/datasets/raw` first. This cell fails on missing images, invalid categories, zero-area boxes, or out-of-bounds boxes.

In [ ]:
!python scripts/prepare_data.py --drive-root "$DRIVE_ROOT" --tracks 2class 10class --validate